# 诗歌生成

# 数据处理

In [17]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets

start_token = 'bos'
end_token = 'eos'

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r', encoding='utf-8') as fd:
        for line in fd:
            outs = line.strip().split(':')
            content = ''.join(outs[1:])
            # 给每首诗补上起始标记 bos 和结束标记 eos，方便模型学习序列边界
            ins = [start_token] + list(content) + [end_token] 
            if len(ins) > 200:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    words = ('PAD', 'UNK') + words[:len(words)]
    word2id = dict(zip(words, range(len(words))))
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset():
    instances, word2id, id2word = process_dataset('./poems.txt')
    # 数据集中的每个样本由两部分组成：一首诗的 id 序列 + 该序列的真实长度
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                            (tf.int64, tf.int64), 
                                            (tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.shuffle(buffer_size=10240)
    # 不同诗句长度不同，因此这里需要按 batch 做 padding 补齐
    ds = ds.padded_batch(100, padded_shapes=(tf.TensorShape([None]),tf.TensorShape([])))
    # 用前 t 个字预测第 t+1 个字，所以输入和标签错开一位
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    return ds, word2id, id2word

# 模型代码， 完成建模代码

In [18]:
class myRNNModel(keras.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)
        # Embedding 层：把离散的字 id 转成连续的向量表示
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        # 这里使用最基础的 SimpleRNNCell 来构建 RNN 层，隐藏状态维度是 128
        self.rnncell = tf.keras.layers.SimpleRNNCell(128)
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)
        # 将隐藏状态映射到整个词表大小，得到每个字的预测分数
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    def call(self, inp_ids):
        '''
        前向传播：
        1. 输入是字的 id 序列，形状为 [batch, seq_len]
        2. 先通过 Embedding 映射为词向量序列
        3. 再送入 RNN，得到每个时间步的隐藏状态
        4. 最后通过全连接层映射到词表大小，得到每个位置预测下一个字的 logits
        '''
        inp_emb = self.embed_layer(inp_ids)  # [batch, seq_len, emb_dim]
        rnn_out = self.rnn_layer(inp_emb)    # [batch, seq_len, hidden_dim]
        logits = self.dense(rnn_out)         # [batch, seq_len, vocab_size]
        return logits
    
    def get_next_token(self, x, state):
        '''
        生成阶段一次只输入一个 token。
        shape(x) = [b_sz,]
        state 表示上一个时间步传下来的隐藏状态。
        '''
    
        inp_emb = self.embed_layer(x) # shape(b_sz, emb_sz)
        h, state = self.rnncell.call(inp_emb, state) # shape(b_sz, h_sz)
        logits = self.dense(h) # shape(b_sz, v_sz)
        out = tf.argmax(logits, axis=-1)
        return out, state

## 一个计算sequence loss的辅助函数，只需了解用途。

In [19]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [20]:
def compute_loss(logits, labels, seqlen):
    # 先计算每个位置的交叉熵
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    # 再根据真实序列长度做平均，忽略 padding 补齐部分带来的干扰
    losses = reduce_avg(losses, seqlen, dim=1)
    return tf.reduce_mean(losses)

def train_one_step(model, optimizer, x, y, seqlen):
    '''
    完成一步训练：前向传播 -> 计算 loss -> 自动求导 -> 参数更新
    '''
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y, seqlen)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(epoch, model, optimizer, ds):
    loss = 0.0
    accuracy = 0.0
    for step, (x, y, seqlen) in enumerate(ds):
        loss = train_one_step(model, optimizer, x, y, seqlen)

        if step % 500 == 0:
            print('epoch', epoch, ': loss', loss.numpy())

    return loss

# 训练优化过程

In [21]:
optimizer = optimizers.Adam(0.0005)
train_ds, word2id, id2word = poem_dataset()
model = myRNNModel(word2id)

# 先用一个 batch 显式调用一次模型，让各层参数完成创建
for sample_x, sample_y, sample_seqlen in train_ds.take(1):
    _ = model(sample_x)

for epoch in range(10):
    loss = train(epoch, model, optimizer, train_ds)

epoch 0 : loss 8.820901


KeyboardInterrupt: 

# 生成过程

In [8]:
def gen_sentence(begin_word=None, max_len=50):
    # SimpleRNNCell 只有一个隐状态，因此这里 state 只需要一个张量
    state = [tf.zeros(shape=(1, 128), dtype=tf.float32)]
    cur_token = tf.constant([word2id['bos']], dtype=tf.int32)
    collect = []

    # 如果指定了起始字，先把起始字送入网络，作为生成的开头
    if begin_word is not None:
        collect.append(begin_word)
        cur_token = tf.constant([word2id.get(begin_word, word2id['UNK'])], dtype=tf.int32)

    for _ in range(max_len):
        cur_token, state = model.get_next_token(cur_token, state)
        next_id = int(cur_token.numpy()[0])
        next_word = id2word[next_id]

        # 生成到 eos 时结束
        if next_word == 'eos':
            break
        if next_word not in ['bos', 'PAD']:
            collect.append(next_word)

    return ''.join(collect)

print(gen_sentence())
print(gen_sentence('日'))
print(gen_sentence('红'))
print(gen_sentence('山'))

NameError: name 'word2id' is not defined